# OCR bill chuyển khoản — TRAIN FROM SCRATCH (Colab)

Notebook train **1 model CRNN+CTC từ đầu** (không pretrain, **không finetune**),
chỉ cho **bill chuyển khoản** ngân hàng/ví (MB, VietinBank, Momo…).

**Kiến trúc OCR mới (tất cả phần học máy đều train from scratch):**
1. Phát hiện dòng chữ bằng xử lý ảnh cổ điển (`app/ocr_detect.py`) — không cần train.
2. Recognizer CRNN+CTC tổng quát (`data/train_ocr_recognizer.py`) — học từ dữ liệu synthetic
   sinh on-the-fly (`data/gen_ocr_lines.py --bank-only`): số tiền, tên, ghi chú, nội dung, footer app NH…
3. Parse bill CK: `app/transfer_parse.py` + `app/ocr_transfer.py`.

**Yêu cầu:** Runtime → Change runtime type → **T4 GPU**.

**Luồng train mới:** CELL 0 → 1 → 2 → **3.5** (đã có `data/real_lines` thì bỏ qua) → **4 train** (lặp khi hết giờ) → 5 → 6 → 7 copy model.

**Upload Drive:** chạy trên máy `python scripts/pack_colab_transfer_train.py` → upload `colab_upload/ai_service.zip` → giải nén `MyDrive/thesis/ai_service/`.

**Colab 2h20 — train nhiều phiên (KHÔNG train một mạch):**
- CELL 4 tự **dừng + lưu checkpoint** sau ~130 phút và sau MỖI epoch.
- Khi hết giờ / bị ngắt: **chạy lại CELL 4** → tự **resume** từ epoch dang dở (checkpoint nằm trên Drive).
- Lặp lại tới khi log báo "Da train du epochs" hoặc early stop.

**Dùng dữ liệu thật của bạn (`data_train/`):** CELL 3.5 tách dòng + tạo `labels.csv` (có nhãn gợi ý để bạn sửa nhanh). Sửa xong, CELL 4 sẽ trộn 30% dòng thật vào mỗi batch.

**Nâng cấp so với bản trước (sửa lỗi nuốt ký tự / mất dấu ở dòng dài):**
- `width_divisor=8` → số bước CTC (T) gấp đôi (lên 128) → đủ "khe" căn ký tự dòng 40-60 ký tự.
- LSTM hidden 384, `max_w=1024`, dropout 0.25.
- Dataset thêm nhiều dòng dài 20-60 ký tự, từ khóa hóa đơn (THÀNH TIỀN/VAT/GIẢM GIÁ...), số hàng nghìn lớn, tiếng Việt có dấu.
- Augmentation thêm motion blur + JPEG compression.
- Hậu xử lý `app/ocr_postprocess.py` tự sửa từ khóa OCR sai (TgNG→TỔNG, VNB→VND...).


In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 0 — Cài thư viện + Mount Google Drive
# ══════════════════════════════════════════════════════════════
import subprocess, sys
from pathlib import Path

def pip(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

for mod, pkg in [("torch", "torch"), ("numpy", "numpy"), ("pandas", "pandas"),
                 ("PIL", "Pillow"), ("matplotlib", "matplotlib")]:
    try:
        __import__(mod)
    except ImportError:
        pip(pkg)

# OpenCV (phát hiện dòng tốt hơn — có fallback numpy nếu thiếu)
try:
    import cv2  # noqa
except ImportError:
    pip("opencv-python-headless")

# Font đa dạng cho synthetic
subprocess.run(["apt-get", "install", "-y", "-qq",
                "fonts-dejavu", "fonts-liberation", "fonts-freefont-ttf", "fonts-noto-core"],
               check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

from google.colab import drive
drive.mount("/content/drive")
print("Drive mounted.")


In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 1 — Tìm thư mục ai_service trên Drive + cấu hình đường dẫn
# ══════════════════════════════════════════════════════════════
from pathlib import Path
import sys

AI_ROOT = None
CANDIDATES = [
    Path("/content/drive/MyDrive/thesis/ai_service"),
    Path("/content/drive/MyDrive/ai_service"),
    Path("/content/drive/MyDrive/Doantotnghiep2/ai_service"),
    Path("/content/ai_service"),
]
for p in CANDIDATES:
    if (p / "app" / "ocr_net.py").is_file():
        AI_ROOT = p
        break
if AI_ROOT is None:
    for cand in Path("/content/drive/MyDrive").rglob("ocr_net.py"):
        if cand.parent.name == "app":
            AI_ROOT = cand.parent.parent
            break
assert AI_ROOT is not None, "Khong tim thay ai_service tren Drive (can co app/ocr_net.py)"

WORK_ROOT  = AI_ROOT
# Lưu THẲNG vào Drive để checkpoint + model KHÔNG MẤT khi Colab ngắt (2h20),
# nhờ vậy chạy lại là RESUME tiếp tục được.
MODELS_DIR   = AI_ROOT / "models"
DRIVE_MODELS = MODELS_DIR
LOG_DIR      = MODELS_DIR / "ocr_logs"
DATA_TRAIN   = AI_ROOT / "data_train"          # ảnh thật bạn tự thu thập (theo merchant)
REAL_DIR     = AI_ROOT / "data" / "real_lines" # dòng thật đã tách + nhãn (tạo ở CELL 3.5)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

if str(WORK_ROOT) not in sys.path:
    sys.path.insert(0, str(WORK_ROOT))

print("AI_ROOT      :", AI_ROOT)
print("MODELS_DIR   :", MODELS_DIR, "(model + checkpoint luu o Drive)")
print("DATA_TRAIN   :", DATA_TRAIN, "->", "CO" if DATA_TRAIN.is_dir() else "KHONG CO")
print("REAL_DIR     :", REAL_DIR)
real_csv = REAL_DIR / "labels.csv"
if real_csv.is_file():
    import pandas as pd
    _df = pd.read_csv(real_csv)
    _n = (_df["text"].astype(str).str.strip() != "").sum()
    print(f"REAL_LINES   : {_n}/{len(_df)} dong co nhan (train tron 30% batch)")
else:
    print("REAL_LINES   : chua co — chay CELL 3.5 hoac upload data/real_lines")


In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 2 — GPU + siêu tham số
# ══════════════════════════════════════════════════════════════
import torch
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if DEVICE.type == "cpu":
    print("CANH BAO: dang dung CPU -> Runtime > Change runtime type > T4 GPU")

EPOCHS      = 28      # early-stop thường dừng sớm hơn
TRAIN_SIZE  = 80000   # số dòng synthetic bank-only mỗi epoch
VAL_SIZE    = 3000
BATCH_SIZE  = 48      # OOM → giảm 32
PATIENCE    = 7
NUM_WORKERS = 4

# ── TRAIN FROM SCRATCH (như cũ — KHÔNG finetune / pretrained) ─────
FROM_SCRATCH = True     # True: random weights, xóa ckpt cũ, KHÔNG --finetune
RESUME       = False    # False khi train mới; True nếu muốn tiếp phiên Colab cũ
BANK_ONLY    = True     # synthetic chỉ bill chuyển khoản
NOTE_FOCUS   = 0.30     # 30% ghi chú ngắn (mung me 8/3, tra no…)
REAL_OVERSAMPLE = 25    # nhân bản dòng thật (95 unique -> ~2375 slot/epoch)
print(f"EPOCHS={EPOCHS} FROM_SCRATCH={FROM_SCRATCH} RESUME={RESUME} "
      f"BANK_ONLY={BANK_ONLY} NOTE_FOCUS={NOTE_FOCUS}")


In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 3 — (Tùy chọn) Xem thử vài dòng synthetic
# ══════════════════════════════════════════════════════════════
import random, sys
if str(WORK_ROOT / "data") not in sys.path:
    sys.path.insert(0, str(WORK_ROOT / "data"))
import matplotlib.pyplot as plt
from gen_ocr_lines import sample_line, render_line

rng = random.Random(7)
fig, axes = plt.subplots(8, 1, figsize=(9, 11))
for ax in axes:
    text, kind = sample_line(rng, bank_only=True)
    img = render_line(text, rng, kind=kind, augment=True)
    ax.imshow(img, cmap="gray"); ax.axis("off")
    ax.set_title(f"[{kind}] {text}", fontsize=9, loc="left")
plt.tight_layout(); plt.show()


In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 3.5 — Dòng thật (bỏ qua nếu đã upload data/real_lines từ máy)
# ══════════════════════════════════════════════════════════════
%cd {WORK_ROOT}
import pandas as pd
csv = REAL_DIR / "labels.csv"
if csv.is_file() and (REAL_DIR / "images").is_dir():
    print("Da co real_lines — bo qua prepare. Chay seeds + hien thong tin.")
    !python data/transfer_label_seeds.py
else:
    USE_PSEUDO = (MODELS_DIR / "ocr_reco_model.pt").is_file()
    pseudo_flag = "--pseudo" if USE_PSEUDO else ""
    if DATA_TRAIN.is_dir():
        !python data/prepare_real_lines.py --src {DATA_TRAIN} --out {REAL_DIR} --models-dir {MODELS_DIR} {pseudo_flag}
        !python data/transfer_label_seeds.py
    else:
        print("Khong thay data_train/ — train chi synthetic.")
if csv.is_file():
    df = pd.read_csv(csv).fillna("")
    n = (df["text"].astype(str).str.strip() != "").sum()
    print(f"\n{len(df)} dong | co text: {n}")
    display(df[df["text"].astype(str).str.strip() != ""].head(25))
    print("\n>>> (Tuy chon) sua them labels.csv tren Drive truoc CELL 4:", csv)


In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 4 — TRAIN có CHECKPOINT/RESUME (Colab 2h20)
# ▶ Tự dừng + lưu checkpoint sau ~130 phút (--max-minutes 130).
# ▶ Bị ngắt / hết giờ: CHẠY LẠI ĐÚNG CELL NÀY -> tự RESUME từ epoch dang dở.
# ▶ Trộn dòng thật (REAL_DIR) vào synthetic nếu labels.csv đã có `text`.
# ▶ Lặp lại cho tới khi log in "Da train du epochs" hoặc early stop.
# ══════════════════════════════════════════════════════════════
%cd {WORK_ROOT}
REAL_RATIO  = 0.35   # 35% batch từ ảnh thật (oversample x25)
MAX_MINUTES = 130    # < 140 phút (2h20) để kịp lưu checkpoint truoc khi Colab cat

real_arg = f"--real-dir {REAL_DIR}" if (REAL_DIR / 'labels.csv').is_file() else ""
scratch_arg = "--from-scratch" if FROM_SCRATCH else ""
resume_arg = "" if RESUME or FROM_SCRATCH else "--no-resume"
bank_arg = "--bank-only" if BANK_ONLY else "--no-bank-only"
!python data/train_ocr_recognizer.py \
    --epochs {EPOCHS} --train-size {TRAIN_SIZE} --val-size {VAL_SIZE} \
    --batch-size {BATCH_SIZE} --patience {PATIENCE} --num-workers {NUM_WORKERS} \
    --models-dir {MODELS_DIR} --log-dir {LOG_DIR} --ckpt-dir {MODELS_DIR} \
    --max-minutes {MAX_MINUTES} --real-ratio {REAL_RATIO} \
    --note-focus {NOTE_FOCUS} --real-oversample {REAL_OVERSAMPLE} \
    {bank_arg} {scratch_arg} {resume_arg} {real_arg}

import json
meta_p = MODELS_DIR / "ocr_reco_meta.json"
ckpt_p = MODELS_DIR / "ocr_reco_ckpt.pt"
if meta_p.is_file():
    m = json.loads(meta_p.read_text(encoding="utf-8"))
    print(f"\nMODEL hien tai: best_val_cer={m.get('best_val_cer')} "
          f"val_exact={m.get('val_exact_acc')} epochs_run={m.get('epochs_run')} "
          f"params={m.get('params_million')}M")
print("Checkpoint:", "CO (resume duoc)" if ckpt_p.is_file() else "chua co")
print(">>> Neu log bao het gio/chua du epoch: CHAY LAI CELL NAY de train tiep.")


In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 5 — Đánh giá: metric + loss curve + bảng dự đoán (synthetic val)
# ══════════════════════════════════════════════════════════════
import pandas as pd, torch
from IPython.display import display
import matplotlib.pyplot as plt
import sys
if str(WORK_ROOT) not in sys.path:
    sys.path.insert(0, str(WORK_ROOT))

from app.ocr_recognizer import load_recognizer_bundle
import importlib, data.train_ocr_recognizer as T
importlib.reload(T)

bundle = load_recognizer_bundle(MODELS_DIR, device=DEVICE) or load_recognizer_bundle(DRIVE_MODELS, device=DEVICE)
assert bundle is not None, "Chua co model — chay CELL 4"

va_loader = torch.utils.data.DataLoader(
    T.SyntheticLineDataset(2000, augment=True, fixed=True, bank_only=True),
    batch_size=64, shuffle=False, collate_fn=T.collate, num_workers=2,
)
metrics, preds, _ = T.evaluate(bundle.model, va_loader, DEVICE)
print("VAL  exact=%.4f  CER=%.4f  WER=%.4f  conf=%.4f"
      % (metrics["exact_acc"], metrics["mean_cer"], metrics["mean_wer"], metrics["mean_confidence"]))

log = LOG_DIR / "reco_epoch_log.csv"
if log.is_file():
    df = pd.read_csv(log)
    fig, ax = plt.subplots(1, 2, figsize=(13, 4))
    ax[0].plot(df["epoch"], df["train_loss"], label="train"); ax[0].plot(df["epoch"], df["val_loss"], label="val")
    ax[0].set_title("CTC loss"); ax[0].legend(); ax[0].grid(alpha=.3)
    ax[1].plot(df["epoch"], df["val_cer"], color="green"); ax[1].set_title("val CER"); ax[1].grid(alpha=.3)
    plt.tight_layout(); plt.show()

display(pd.DataFrame(preds).head(30))


In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 6 — TEST ẢNH THẬT: detect dòng -> recognize -> parse field
# ══════════════════════════════════════════════════════════════
from google.colab import files
from PIL import Image, ImageDraw
from io import BytesIO
import matplotlib.pyplot as plt
import sys
if str(WORK_ROOT) not in sys.path:
    sys.path.insert(0, str(WORK_ROOT))

from app.ocr_recognizer import load_recognizer_bundle, recognize_batch
from app.ocr_detect import detect_lines, cv2_available
from app.transfer_parse import parse_transfer_image

# classify (tùy chọn) để ra danh mục
cls = None
try:
    from app.classify_infer import load_classify_bundle
    cls = load_classify_bundle(AI_ROOT / "models")
except Exception as e:
    print("classify khong san sang:", e)

bundle = load_recognizer_bundle(MODELS_DIR, device=DEVICE) or load_recognizer_bundle(DRIVE_MODELS, device=DEVICE)
assert bundle is not None, "Chua co model — chay CELL 4"
print("OpenCV detect:", cv2_available())

print("Chon anh bill chuyen khoan:")
up = files.upload()
for name, raw in up.items():
    img = Image.open(BytesIO(raw)); img = img.convert("RGB") if img.mode not in ("L","RGB") else img

    # 1) Detect + recognize tung dong (xem chi tiet)
    lbs = detect_lines(img)
    preds = recognize_batch(bundle, [lb.image for lb in lbs])
    vis = img.convert("RGB").copy(); d = ImageDraw.Draw(vis)
    for lb in lbs:
        d.rectangle([lb.x0, lb.y0, lb.x1, lb.y1], outline=(255,0,0), width=2)
    plt.figure(figsize=(7, 10)); plt.imshow(vis); plt.axis("off"); plt.title(name); plt.show()
    print(f"\n--- {len(lbs)} dong nhan dang ---")
    for (t, c) in preds:
        print(f"  [{c:.2f}] {t}")

    # 2) Parse field
    res = parse_transfer_image(img, bundle, classify=cls)
    print("\n=== KET QUA PARSE (transfer-only) ===")
    print("  amount_vnd :", res.amount_vnd)
    print("  date       :", res.transaction_date)
    print("  sender     :", res.sender)
    print("  receiver   :", res.receiver)
    print("  note       :", res.note)
    print("  category   :", res.category, "| type:", res.type)
    print("  needs_review:", res.needs_review)


In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 6B — ĐÁNH GIÁ ẢNH THẬT bill chuyển khoản
# ══════════════════════════════════════════════════════════════
# Chuẩn bị: AI_ROOT/data/transfer_test/ (ảnh CK + labels.csv)
# Nhãn: labels.csv (filename,amount,date,sender,receiver,note)
#       HOẶC sidecar JSON cùng tên ảnh (anh01.jpg -> anh01.json)
%cd {WORK_ROOT}
!python data/evaluate_real.py --models-dir {DRIVE_MODELS} --test-dir {AI_ROOT}/data

# Xem chi tiết dự đoán từng ảnh:
import pandas as pd
from pathlib import Path
rep = DRIVE_MODELS / "ocr_logs" / "real_eval"
for f in ("real_eval_receipt.csv", "real_eval_transfer.csv"):
    p = rep / f
    if p.is_file():
        print("\n===", f, "===")
        display(pd.read_csv(p))

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 7 — Kiểm tra artifact trên Drive (đã lưu sẵn ở MODELS_DIR)
# ══════════════════════════════════════════════════════════════
# MODELS_DIR đã nằm trên Drive nên model + checkpoint tự lưu sẵn, không cần copy.
import json
need = ["ocr_reco_model.pt", "ocr_reco_meta.json", "ocr_reco_charset.json"]
print("Thu muc service đọc model:", MODELS_DIR)
for name in need:
    p = MODELS_DIR / name
    print(f"  {'[OK]' if p.is_file() else '[THIEU]'} {name}")
ckpt = MODELS_DIR / "ocr_reco_ckpt.pt"
print(f"  {'[OK]' if ckpt.is_file() else '[--]'} ocr_reco_ckpt.pt (checkpoint resume)")
mp = MODELS_DIR / "ocr_reco_meta.json"
if mp.is_file():
    m = json.loads(mp.read_text(encoding="utf-8"))
    print(f"\nbest_val_cer={m.get('best_val_cer')} val_exact={m.get('val_exact_acc')} "
          f"epochs_run={m.get('epochs_run')} real_mixed={m.get('real_mixed')}")
print("\nDU 3 FILE -> copy ve may: ocr_reco_model.pt + meta + charset")
print("Muon train tiep Colab: FROM_SCRATCH=False, RESUME=True, chay lai CELL 4.")
